# 🛒 Olist Brazilian E-Commerce — EDA & Order Summary Report

**Project:** Customer 360 & Marketing Attribution Platform
**Dataset:** Brazilian E-Commerce Public Dataset by Olist (9 CSV files in `Raw Data/`)

**Outputs:**
- `Olist_Order_Summary_Report.xlsx` (4 sheets)
- `Summary Data/order_summary.csv` (clean, delivered orders only)

---

### 🗺️ Notebook Flow
```
 PART A — EXPLORE                         PART B — BUILD & EXPORT
 ───────────────────                      ──────────────────────────────────────
 1. Setup                                 5. Load orders & parse dates
 2. First look: customers        ──►      6. Feature engineering
 3. Health check: all 9 files             7. Order summary table
 4. Deep dive: geolocation                8. Report sheets (overview, dictionary, quality)
                                          9. Export Excel report
                                         10. Clean & validate
                                         11. Export clean CSV
                                         12. Key findings & next steps
```

| Part | Goal |
|------|------|
| **A. Exploratory Data Analysis** | Understand the raw data: size, duplicates, missing values, data types |
| **B. Order Summary Report** | Turn the orders table into a clean, documented, analysis-ready dataset |

## 1. Setup — Libraries, Paths & Constants

| Library | Used for |
|---------|----------|
| `pandas` | Reading and analysing tables |
| `pathlib` | File paths that work on Windows, Mac and Linux |
| `openpyxl` | Writing the Excel report (used by pandas behind the scenes) |

> **Note:** If `openpyxl` isn't installed yet, uncomment the `%pip` line and run it **once**, then restart the kernel.

In [ ]:
# %pip install openpyxl     # ← run once if openpyxl is missing

# --- Core libraries ---
import pandas as pd                 # data manipulation
from pathlib import Path            # OS-independent file paths

# --- Display settings: show every column when printing wide tables ---
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# --- Paths ---
DATA_DIR     = Path("Raw Data")                         # input folder (raw CSVs)
SUMMARY_DIR  = Path("Summary Data")                     # output folder for clean CSVs
EXCEL_REPORT = Path("Olist_Order_Summary_Report.xlsx")  # output Excel workbook
SUMMARY_DIR.mkdir(exist_ok=True)                        # create output folder if needed

# --- Constants ---
SECONDS_PER_DAY  = 86_400
SECONDS_PER_HOUR = 3_600

# Quick sanity check: list every CSV we are about to analyse
csv_files = sorted(DATA_DIR.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files in '{DATA_DIR}':")
for f in csv_files:
    print("  •", f.name)

---
# 🅰️ PART A — Exploratory Data Analysis

Before building anything, we answer four questions about the raw data:
1. What files do we have and how big are they?
2. Are there **missing values** or **duplicate rows**?
3. Are the **data types** correct (e.g. dates stored as text)?
4. Which tables need cleaning before they can be joined?

## 2. First Look — Customers Table

Let's start with a single file to get a feel for the structure.
Each row represents **one customer per order**:

- `customer_id` → changes with every order (key to join with *orders*)
- `customer_unique_id` → the **real** person; the same person can have several `customer_id`s

In [ ]:
# Load the customers dataset
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")

# Preview the first 5 rows
customers.head()

In [ ]:
# How many real (unique) customers vs. customer IDs?
# A gap between these two numbers = repeat buyers.
total_ids    = customers["customer_id"].nunique()
unique_people = customers["customer_unique_id"].nunique()

print(f"customer_id (one per order) : {total_ids:,}")
print(f"customer_unique_id (people) : {unique_people:,}")
print(f"Repeat-purchase IDs         : {total_ids - unique_people:,}")

## 3. Data Health Check — All Files

Instead of inspecting each file manually, we loop over every CSV and report:

| Check | Why it matters |
|-------|----------------|
| **Shape** | Size of the table (rows × columns) |
| **Duplicate rows** | Fully repeated rows inflate counts and sums |
| **Missing values** | Gaps we must handle before analysis |
| **Data types** | Dates read as text must be converted |

The results are also collected into a single **overview table** for easy comparison.

In [ ]:
health_rows = []   # one summary row per file

for path in csv_files:
    df = pd.read_csv(path, low_memory=False)

    # ---------- Detailed printout ----------
    print(f"\n{'=' * 60}")
    print("📄 File:", path.name)
    print("Shape         :", df.shape)
    print("Duplicate rows:", df.duplicated().sum())

    # Show only columns that actually have missing values
    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    print("Missing values:", "None ✅" if missing.empty else "")
    if not missing.empty:
        print(missing.to_string())

    print("Data types:")
    print(df.dtypes.to_string())

    # ---------- Collect a compact summary ----------
    health_rows.append({
        "file": path.name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": int(df.duplicated().sum()),
        "total_missing_cells": int(df.isna().sum().sum()),
        "columns_with_missing": int((df.isna().sum() > 0).sum()),
    })

### 📊 Health Check Overview
A side-by-side view of every file — this makes the problem areas jump out immediately.

In [ ]:
health_overview = (
    pd.DataFrame(health_rows)
    .sort_values("rows", ascending=False)
    .reset_index(drop=True)
)
health_overview

> **💡 What stands out**
> - **Geolocation** has ~262K fully duplicated rows → needs de-duplication.
> - **Reviews** are missing most comment titles/messages → expected (comments are optional).
> - **Orders** are missing some approval/delivery dates → linked to undelivered or cancelled orders.
> - **Products** have 610 rows without a category → will appear as "unknown".
> - All **date columns are stored as text** → must be converted with `pd.to_datetime`.

## 4. Deep Dive — Geolocation Duplicates

The geolocation file has over **1 million rows** but many are exact copies.
Let's look at a few to confirm they are genuine duplicates (same ZIP, same coordinates).

In [ ]:
geo = pd.read_csv(DATA_DIR / "olist_geolocation_dataset.csv")

# Show a sample of rows that are exact duplicates of an earlier row
display(geo[geo.duplicated()].head())

In [ ]:
# How much smaller does the table get after removing exact duplicates?
geo_dedup = geo.drop_duplicates()

print(f"Original rows     : {len(geo):,}")
print(f"After de-dup      : {len(geo_dedup):,}")
print(f"Rows removed      : {len(geo) - len(geo_dedup):,} "
      f"({(1 - len(geo_dedup) / len(geo)) * 100:.1f}%)")

# Even after de-dup, one ZIP prefix can have several coordinates.
# For joins we will later need ONE row per ZIP prefix (e.g. the mean lat/lng).
print(f"Unique ZIP prefixes: {geo_dedup['geolocation_zip_code_prefix'].nunique():,}")

---
# 🅱️ PART B — Order Summary Report

The **orders** table is the heart of the dataset — every other table joins to it.
In this part we turn it into a clean, documented, analysis-ready table and export it.

```
 Raw CSV ──► 5. Parse dates ──► 6. Features ──► 7. Summary table ──► 8. Report sheets
                                                                           │
            11. Export CSV ◄── 10. Clean & validate ◄── 9. Export Excel ◄──┘
```

## 5. Load Data & Parse Dates

All timestamp columns arrive as **text**. Converting them to `datetime` lets us
subtract dates, extract months/weekdays, and filter by time.

In [ ]:
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")

date_columns = [
    "order_purchase_timestamp",       # customer placed the order
    "order_approved_at",              # payment approved
    "order_delivered_carrier_date",   # handed to logistics partner
    "order_delivered_customer_date",  # customer received it
    "order_estimated_delivery_date",  # promised delivery date
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")  # bad values → NaT

print(f"Loaded {len(orders):,} orders")
orders[date_columns].dtypes

### 5.1 Order status distribution
Most missing delivery dates come from orders that never reached the customer — this breakdown shows how many of each.

In [ ]:
# Distribution of order statuses — explains most of the missing delivery dates
orders["order_status"].value_counts()

## 6. Feature Engineering

We derive new columns that answer common business questions:

| Group | Column | Business question |
|-------|--------|-------------------|
| 🗓️ Time | `purchase_year`, `purchase_month`, `purchase_day` | *When* do customers buy? |
| ⏱️ Approval | `approval_hours` | How fast is payment approved? |
| 🚚 Delivery | `delivery_days` | How long does the customer wait? |
| 🚚 Delivery | `delivery_variance_days` | Did we beat or miss the promise? |
| ✅ Status | `is_delivered` | Was the order completed? |
| 🏷️ Label | `delivery_performance` | Early / On Time / Late / Not Delivered |

### 6.1 Time-based features

In [ ]:
purchase_ts = orders["order_purchase_timestamp"]

orders["purchase_year"]  = purchase_ts.dt.year                                 # e.g. 2017
orders["purchase_month"] = purchase_ts.dt.to_period("M").astype("string")     # e.g. "2017-10"
orders["purchase_day"]   = purchase_ts.dt.day_name()                           # e.g. "Monday"

### 6.2 Approval & delivery durations

In [ ]:
# Hours from purchase to payment approval
orders["approval_hours"] = (
    orders["order_approved_at"] - orders["order_purchase_timestamp"]
).dt.total_seconds() / SECONDS_PER_HOUR

# Days from purchase to customer delivery (customer waiting time)
orders["delivery_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.total_seconds() / SECONDS_PER_DAY

# Days between actual and promised delivery  →  positive = late, negative = early
orders["delivery_variance_days"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.total_seconds() / SECONDS_PER_DAY

### 6.3 Delivery status & performance label

> **🛠️ Improvement over the previous version:** the old logic compared the *exact* time difference
> to `0`, so **"On Time" was almost never assigned** (a difference of exactly 0.000 seconds is very rare).
> The estimated date has no time component, so we now compare **calendar dates**:
> delivered on the promised day = *On Time*.

In [ ]:
# Simple flag: did the order reach "delivered" status?
orders["is_delivered"] = orders["order_status"].eq("delivered")

# Default label for every order
orders["delivery_performance"] = "Not Delivered"

# Only orders with both an actual and an estimated date can be classified
has_dates = (
    orders["order_delivered_customer_date"].notna()
    & orders["order_estimated_delivery_date"].notna()
)

# Compare calendar days (drop the time-of-day part)
actual_day   = orders["order_delivered_customer_date"].dt.normalize()
promised_day = orders["order_estimated_delivery_date"].dt.normalize()

orders.loc[has_dates & (actual_day <  promised_day), "delivery_performance"] = "Early"
orders.loc[has_dates & (actual_day == promised_day), "delivery_performance"] = "On Time"
orders.loc[has_dates & (actual_day >  promised_day), "delivery_performance"] = "Late"

# Check the result
orders["delivery_performance"].value_counts()

## 7. Build the Order Summary Table

We keep only the columns that matter for reporting, in a logical left-to-right order:
**IDs → status → purchase time → approval → logistics → delivery metrics → labels**.

In [ ]:
summary_columns = [
    # --- Identifiers ---
    "order_id", "customer_id",
    # --- Status ---
    "order_status",
    # --- Purchase timing ---
    "order_purchase_timestamp", "purchase_year", "purchase_month", "purchase_day",
    # --- Approval ---
    "order_approved_at", "approval_hours",
    # --- Logistics milestones ---
    "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date",
    # --- Delivery metrics & labels ---
    "delivery_days", "delivery_variance_days", "is_delivered", "delivery_performance",
]

order_summary = orders[summary_columns].copy()
print("Order summary shape:", order_summary.shape)
order_summary.head()

## 8. Build the Report Sheets

The Excel workbook contains 4 sheets, each serving a different reader:

| Sheet | Audience | Content |
|-------|----------|---------|
| **Order Summary** | Analysts | Row-level order data |
| **Dataset Overview** | Managers | Headline KPIs + status breakdown |
| **Column Dictionary** | Everyone | What each column means |
| **Data Quality** | Data team | Missing / unique values per column |

### 8.1 Dataset Overview (headline KPIs)

In [ ]:
overview_data = [
    # [Metric, Value, Description]
    ["Total Records",            len(orders),                                   "Total number of order records"],
    ["Unique Orders",            orders["order_id"].nunique(),                  "Number of distinct orders"],
    ["Unique Customers",         orders["customer_id"].nunique(),               "Distinct customer IDs represented in orders"],
    ["Duplicate Order IDs",      orders["order_id"].duplicated().sum(),         "Repeated order IDs"],
    ["First Purchase Date",      orders["order_purchase_timestamp"].min(),      "Earliest purchase timestamp"],
    ["Last Purchase Date",       orders["order_purchase_timestamp"].max(),      "Latest purchase timestamp"],
    ["Delivered Orders",         orders["order_status"].eq("delivered").sum(),  "Orders with delivered status"],
    ["Cancelled Orders",         orders["order_status"].eq("canceled").sum(),   "Orders with canceled status"],
    ["Average Delivery Days",    round(orders["delivery_days"].mean(), 2),      "Average purchase-to-delivery duration"],
    ["Median Delivery Days",     round(orders["delivery_days"].median(), 2),    "Median purchase-to-delivery duration"],
    ["Average Delivery Variance",round(orders["delivery_variance_days"].mean(), 2), "Positive means late; negative means early"],
    ["Late Delivery Rate (%)",   round(orders.loc[has_dates, "delivery_performance"].eq("Late").mean() * 100, 2),
                                                                                "Share of delivered orders that arrived late"],
    ["Missing Purchase Dates",   orders["order_purchase_timestamp"].isna().sum(),       "Records without purchase timestamps"],
    ["Missing Customer Delivery Dates", orders["order_delivered_customer_date"].isna().sum(), "Records without actual customer delivery dates"],
]

dataset_overview = pd.DataFrame(overview_data, columns=["Metric", "Value", "Description"])

# ---- Append the order-status breakdown underneath the KPIs ----
status_counts = (
    orders["order_status"]
    .value_counts(dropna=False)
    .rename_axis("Metric")
    .reset_index(name="Value")
    .assign(Description="Number of orders with this status")
)

dataset_overview = pd.concat([dataset_overview, status_counts], ignore_index=True)
dataset_overview

### 8.2 Column Dictionary

In [ ]:
# Plain-English definition for every column in the Order Summary sheet
column_dictionary = pd.DataFrame([
    ["order_id",                      "Unique order identifier",                                   "ID"],
    ["customer_id",                   "Customer identifier linked to the customer dataset",       "ID"],
    ["order_status",                  "Current order status",                                      "Category"],
    ["order_purchase_timestamp",      "Date and time when the order was placed",                   "Datetime"],
    ["purchase_year",                 "Year in which the order was placed",                        "Date"],
    ["purchase_month",                "Year and month of purchase",                                "Date"],
    ["purchase_day",                  "Day of the week when the order was placed",                 "Date"],
    ["order_approved_at",             "Date and time when payment was approved",                   "Datetime"],
    ["approval_hours",                "Hours between purchase and approval",                       "Numeric"],
    ["order_delivered_carrier_date",  "Date when the order was handed to the carrier",             "Datetime"],
    ["order_delivered_customer_date", "Actual date of customer delivery",                          "Datetime"],
    ["order_estimated_delivery_date", "Estimated customer delivery date",                          "Datetime"],
    ["delivery_days",                 "Actual days between purchase and customer delivery",        "Numeric"],
    ["delivery_variance_days",        "Actual minus estimated delivery date; positive means late", "Numeric"],
    ["is_delivered",                  "True when order status is delivered",                       "Boolean"],
    ["delivery_performance",          "Early, On Time, Late (by calendar day) or Not Delivered",   "Category"],
], columns=["Column Name", "Business Meaning", "Data Type"])

# Safety check: every summary column should be documented
missing_docs = set(summary_columns) - set(column_dictionary["Column Name"])
assert not missing_docs, f"Undocumented columns: {missing_docs}"
column_dictionary

### 8.3 Data Quality Report

In [ ]:
# One row per column: type, how much is missing, how many distinct values
data_quality = pd.DataFrame({
    "Column Name":        order_summary.columns,
    "Data Type":          order_summary.dtypes.astype(str).values,
    "Missing Values":     order_summary.isna().sum().values,
    "Missing Percentage": (order_summary.isna().mean().values * 100).round(2),
    "Unique Values":      order_summary.nunique(dropna=True).values,
})

# Show the most incomplete columns first
data_quality.sort_values("Missing Values", ascending=False)

## 9. Export the Excel Report

All four tables are written to a single workbook — one sheet each.

In [ ]:
report_sheets = {
    "Order Summary":     order_summary,
    "Dataset Overview":  dataset_overview,
    "Column Dictionary": column_dictionary,
    "Data Quality":      data_quality,
}

with pd.ExcelWriter(EXCEL_REPORT, engine="openpyxl") as writer:
    for sheet_name, table in report_sheets.items():
        table.to_excel(writer, sheet_name=sheet_name, index=False)

print("✅ Excel report created:", EXCEL_REPORT.resolve())
for name, table in report_sheets.items():
    print(f"   • {name:<18} {len(table):>7,} rows")

## 10. Clean & Validate the Summary Table

For delivery analysis we need **complete** rows. Dropping rows with any missing value
effectively keeps **delivered orders with a full timeline**.

> ⚠️ **Heads-up:** this removes cancelled / unavailable / in-transit orders (~3K rows).
> That's fine for *delivery-time* analysis, but use the full `orders` table if you need
> cancellation rates or total order counts.

In [ ]:
rows_before = len(order_summary)

# Keep only rows with no missing values
order_summary_clean = order_summary.dropna().reset_index(drop=True)

rows_after = len(order_summary_clean)
print(f"Rows before cleaning : {rows_before:,}")
print(f"Rows after cleaning  : {rows_after:,}")
print(f"Rows removed         : {rows_before - rows_after:,}")

# Which statuses were dropped?
print("\nStatuses removed:")
print(order_summary.loc[order_summary.isna().any(axis=1), "order_status"].value_counts().to_string())

### 10.1 Validation checks

In [ ]:
print("--- Shape ---")
print(order_summary_clean.shape)

print("\n--- Missing values (should all be 0) ---")
print(order_summary_clean.isna().sum().sum())

print("\n--- Duplicate rows (should be 0) ---")
print(order_summary_clean.duplicated().sum())

print("\n--- Delivery performance mix ---")
print(order_summary_clean["delivery_performance"].value_counts(normalize=True).mul(100).round(2).to_string())

# Automated guards — the notebook stops here if something is wrong
assert order_summary_clean.isna().sum().sum() == 0, "Missing values remain!"
assert order_summary_clean["order_id"].is_unique, "Duplicate order IDs found!"

In [ ]:
# Descriptive statistics for the numeric & date columns
order_summary_clean.describe()

## 11. Export the Clean CSV

The clean table is saved for downstream use (Power BI dashboards, customer-360 joins, modelling).

In [ ]:
csv_path = SUMMARY_DIR / "order_summary.csv"
order_summary_clean.to_csv(csv_path, index=False)

print("✅ Saved:", csv_path.resolve())
print(f"   {len(order_summary_clean):,} rows × {order_summary_clean.shape[1]} columns")

---
## 12. Key Findings & Next Steps

### 🔍 From the EDA (Part A)
- **9 files**, all loading cleanly; most have **no duplicates** and few missing values.
- **Geolocation** has ~262K exact duplicate rows → de-duplicate and aggregate to one row per ZIP prefix before joining.
- **Reviews** are missing most comment text → expected, comments are optional.
- **Products**: 610 rows without a category → treat as "unknown".
- **All date columns are stored as text** → always convert with `pd.to_datetime`.

### 📦 From the Order Summary (Part B)
- Missing delivery dates belong to **orders that were never delivered** (cancelled, unavailable, in transit…).
- Orders usually arrive **well before** the estimated date — Olist's delivery promises are conservative.
- Delivery performance is labelled **Early / On Time / Late** by calendar day.

### 📁 Outputs
| Output | Location | Rows |
|--------|----------|------|
| Excel report (4 sheets) | `Olist_Order_Summary_Report.xlsx` | all orders |
| Clean order table | `Summary Data/order_summary.csv` | delivered orders only |

### ➡️ Next steps
- Join **customers** (`customer_id`) to add city/state and `customer_unique_id`
- Join **payments** and **order items** to add revenue per order
- Join **reviews** to link delivery performance with customer satisfaction